In [1]:
"""
ST422 Brief 8 — Data Preparation Pipeline
Loads raw STATS19 files, filters to the configured year window, appends
provisional data, cleans coded fields, applies label maps, builds a combined
casualties + vehicles + collision-context table, and writes four CSVs to Cleaned/.

See README.md for path setup and how to change the year window.

Inputs:
  Data/ — raw DfT STATS19 CSVs (see README.md for filenames)

Outputs:
  Cleaned/collisions_clean.csv
  Cleaned/casualties_clean.csv
  Cleaned/vehicles_clean.csv
  Cleaned/cas_full.csv

Run order: Run this notebook first, then ST422_Analysis_v2.ipynb.
Restart kernel before a full re-run to guarantee a clean state.
"""


'\nST422 Brief 8 — Data Preparation Pipeline\nLoads raw STATS19 files, filters to the configured year window, appends\nprovisional data, cleans coded fields, applies label maps, builds a combined\ncasualties + vehicles + collision-context table, and writes four CSVs to Cleaned/.\n\nSee README.md for path setup and how to change the year window.\n\nInputs:\n  Data/ — raw DfT STATS19 CSVs (see README.md for filenames)\n\nOutputs:\n  Cleaned/collisions_clean.csv\n  Cleaned/casualties_clean.csv\n  Cleaned/vehicles_clean.csv\n  Cleaned/cas_full.csv\n\nRun order: Run this notebook first, then ST422_Analysis_v2.ipynb.\nRestart kernel before a full re-run to guarantee a clean state.\n'

In [2]:

# ── Config ────────────────────────────────────────────────────────────────────
# No path editing required — all paths are relative to the project root.
# To change the year window or provisional year, edit the constants below.

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import os

NOTEBOOK_DIR = os.getcwd()
DATA_DIR     = os.path.join(NOTEBOOK_DIR, 'Data')
OUTPUT_DIR   = os.path.join(NOTEBOOK_DIR, 'Cleaned')

# Year window for the historical files.
# YEAR_START: first year to include.
# YEAR_END  : last confirmed published year (update when DfT releases a new annual file).
YEAR_START = 2014
YEAR_END   = 2024

# Set to True to append the provisional year file; False to use confirmed years only.
INCLUDE_PROVISIONAL = True
PROVISIONAL_YEAR    = 2025   # update if a new provisional file is released

print('DATA_DIR  :', DATA_DIR,   '| exists:', os.path.exists(DATA_DIR))
print('OUTPUT_DIR:', OUTPUT_DIR, '| exists:', os.path.exists(OUTPUT_DIR))



DATA_DIR  : /dcs/large/u2207745/ST422_Team1_Project/Final_Workflow/Data_Prep/Data | exists: True
OUTPUT_DIR: /dcs/large/u2207745/ST422_Team1_Project/Final_Workflow/Data_Prep/Cleaned | exists: False


In [3]:

# ── 1. Load raw files ─────────────────────────────────────────────────────────
# Historical files (1979–latest published year) are read and immediately
# filtered to [YEAR_START, YEAR_END]. If INCLUDE_PROVISIONAL is True, the
# provisional year file is appended and tagged with a provisional flag column.


# File paths — built from DATA_DIR; do not edit these directly.
# To point at different files, change DATA_DIR in the Config cell above.
COLLISION_HIST = os.path.join(DATA_DIR, 'dft-road-casualty-statistics-collision-1979-latest-published-year.csv')
CASUALTY_HIST  = os.path.join(DATA_DIR, 'dft-road-casualty-statistics-casualty-1979-latest-published-year.csv')
VEHICLE_HIST   = os.path.join(DATA_DIR, 'dft-road-casualty-statistics-vehicle-1979-latest-published-year.csv')

COLLISION_PROV = os.path.join(DATA_DIR, f'dft-road-casualty-statistics-collision-provisional-{PROVISIONAL_YEAR}.csv')
CASUALTY_PROV  = os.path.join(DATA_DIR, f'dft-road-casualty-statistics-casualty-provisional-{PROVISIONAL_YEAR}.csv')
VEHICLE_PROV   = os.path.join(DATA_DIR, f'dft-road-casualty-statistics-vehicle-provisional-{PROVISIONAL_YEAR}.csv')

LA_LOOKUP = os.path.join(DATA_DIR, 'local-authority-ons-district-names.csv')

DTYPE = {'collision_index': str, 'collision_ref_no': str}

# Load historical files and filter to configured window.
col_hist = pd.read_csv(COLLISION_HIST, dtype=DTYPE, low_memory=False)
cas_hist = pd.read_csv(CASUALTY_HIST,  dtype=DTYPE, low_memory=False)
veh_hist = pd.read_csv(VEHICLE_HIST,   dtype=DTYPE, low_memory=False)

for df in [col_hist, cas_hist, veh_hist]:
    df.columns = df.columns.str.lower()

col_hist = col_hist[col_hist['collision_year'].between(YEAR_START, YEAR_END)].copy()
cas_hist = cas_hist[cas_hist['collision_year'].between(YEAR_START, YEAR_END)].copy()
veh_hist = veh_hist[veh_hist['collision_year'].between(YEAR_START, YEAR_END)].copy()

print(f'Historical loaded — col: {len(col_hist):,}  cas: {len(cas_hist):,}  veh: {len(veh_hist):,}')

# Provisional year appended (if enabled) and tagged so downstream cells can
# filter it out when needed.
if INCLUDE_PROVISIONAL:
    col_prov = pd.read_csv(COLLISION_PROV, dtype=DTYPE, low_memory=False)
    cas_prov = pd.read_csv(CASUALTY_PROV,  dtype=DTYPE, low_memory=False)
    veh_prov = pd.read_csv(VEHICLE_PROV,   dtype=DTYPE, low_memory=False)
    for df in [col_prov, cas_prov, veh_prov]:
        df.columns = df.columns.str.lower()
        df['provisional'] = True
    col = pd.concat([col_hist, col_prov], ignore_index=True)
    cas = pd.concat([cas_hist, cas_prov], ignore_index=True)
    veh = pd.concat([veh_hist, veh_prov], ignore_index=True)
    print(f'Provisional {PROVISIONAL_YEAR} appended — col: {len(col_prov):,}  cas: {len(cas_prov):,}  veh: {len(veh_prov):,}')
else:
    col, cas, veh = col_hist, cas_hist, veh_hist
    print('Provisional data not included.')

for df in [col, cas, veh]:
    df['provisional'] = df['provisional'].fillna(False) if 'provisional' in df.columns else False

print(f'\nFull dataset — col: {len(col):,}  cas: {len(cas):,}  veh: {len(veh):,}')
print(f'Years: {sorted(col["collision_year"].dropna().unique().astype(int).tolist())}')



Historical loaded — col: 1,296,627  cas: 1,687,320  veh: 2,381,280
Provisional 2025 appended — col: 48,472  cas: 60,991  veh: 87,805

Full dataset — col: 1,345,099  cas: 1,748,311  veh: 2,469,085
Years: [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


In [4]:

# ── 2. Clean coded fields ─────────────────────────────────────────────────────
# Invalid codes (-1) are replaced with NaN across categorical fields.
# Longitude is excluded — negative values are valid coordinates for GB.

CODED_COL = [
    'collision_severity', 'enhanced_severity_collision', 'speed_limit', 'road_type',
    'junction_detail', 'junction_control', 'light_conditions', 'urban_or_rural_area',
    'police_force', 'collision_injury_based', 'day_of_week',
    'weather_conditions', 'road_surface_conditions',
]
CODED_CAS = [
    'casualty_severity', 'enhanced_casualty_severity', 'casualty_type',
    'casualty_class', 'age_of_casualty', 'age_band_of_casualty',
    'sex_of_casualty', 'casualty_imd_decile', 'casualty_injury_based',
]
CODED_VEH = [
    'vehicle_type', 'sex_of_driver', 'age_of_driver', 'age_band_of_driver',
    'journey_purpose_of_driver', 'driver_imd_decile', 'propulsion_code',
]

for fields, df in [(CODED_COL, col), (CODED_CAS, cas), (CODED_VEH, veh)]:
    for f in fields:
        if f in df.columns:
            df[f] = pd.to_numeric(df[f], errors='coerce')
            df.loc[df[f] < 0, f] = np.nan

print('Coded fields cleaned — -1 replaced with NaN.')



Coded fields cleaned — -1 replaced with NaN.


In [10]:

# ── 3. Derived columns and label maps ─────────────────────────────────────────
# Adds ksi, fatal, month, human-readable label columns, and joins the LA name
# lookup onto col.

col['date']  = pd.to_datetime(col['date'], dayfirst=True, errors='coerce')
col['month'] = col['date'].dt.month
col['ksi']   = col['collision_severity'].isin([1, 2])
col['fatal'] = col['collision_severity'] == 1

GROUPED_MAP = {
    1: 'Pedestrian', 2: 'Cyclist',
    3: 'Motorcycle rider', 4: 'Motorcycle rider', 5: 'Motorcycle rider',
    23: 'Motorcycle rider', 97: 'Motorcycle rider',
    8: 'Taxi/PHV occupant', 9: 'Car occupant',
    10: 'Minibus/bus occupant', 11: 'Minibus/bus occupant',
    19: 'Van occupant', 20: 'HGV occupant', 21: 'HGV occupant',
    16: 'Other', 17: 'Other', 18: 'Other', 22: 'Other', 90: 'Other',
    98: 'Unknown', 99: 'Unknown',
}
URBAN_MAP    = {1: 'Urban', 2: 'Rural', 3: 'Unallocated'}
ROAD_MAP     = {1: 'Roundabout', 2: 'One way street', 3: 'Dual carriageway',
                6: 'Single carriageway', 7: 'Slip road', 9: 'Unknown', 12: 'One way/Slip road'}
JUNCTION_MAP = {
    0: 'Not at junction', 13: 'T or Y junction', 16: 'Crossroads',
    17: 'More than 4 arms', 18: 'Private drive or entrance',
    19: 'Other junction', 99: 'Unknown'
}
DAY_MAP   = {1: 'Sun', 2: 'Mon', 3: 'Tue', 4: 'Wed', 5: 'Thu', 6: 'Fri', 7: 'Sat'}
FORCE_MAP = {
    1: 'Metropolitan Police', 3: 'Cumbria', 4: 'Lancashire', 5: 'Merseyside',
    6: 'Greater Manchester', 7: 'Cheshire', 10: 'Humberside', 11: 'West Yorkshire',
    12: 'South Yorkshire', 13: 'West Midlands', 14: 'East Midlands', 16: 'Staffordshire',
    17: 'West Mercia', 20: 'Thames Valley', 21: 'Hampshire', 22: 'South East',
    23: 'Essex', 24: 'Norfolk', 25: 'Suffolk', 26: 'Bedfordshire',
    27: 'Hertfordshire', 30: 'Cambridgeshire', 31: 'Northamptonshire',
    32: 'Leicestershire', 33: 'Warwickshire', 34: 'Avon and Somerset',
    35: 'Gloucestershire', 36: 'Wiltshire', 37: 'Dorset', 38: 'Devon and Cornwall',
    39: 'Sussex', 40: 'Kent', 41: 'Surrey', 42: 'Northumbria', 43: 'Durham',
    44: 'North Yorkshire', 45: 'Dyfed-Powys', 46: 'Gwent', 47: 'South Wales',
    48: 'North Wales', 50: 'Lincolnshire', 52: 'Cleveland', 53: 'City of London',
    54: 'Nottinghamshire', 55: 'Derbyshire', 99: 'Unknown/Other'
}

cas['road_user']       = cas['casualty_type'].map(GROUPED_MAP).fillna('Unknown')
col['ur_label']        = col['urban_or_rural_area'].map(URBAN_MAP)
col['road_type_label'] = col['road_type'].map(ROAD_MAP)
col['junction_label']  = col['junction_detail'].map(JUNCTION_MAP)
col['day_label']       = col['day_of_week'].map(DAY_MAP)
col['force_name']      = col['police_force'].map(FORCE_MAP).fillna('Unknown')

la_lookup = (
    pd.read_csv(LA_LOOKUP)[['code', 'la_name']]
    .rename(columns={'code': 'local_authority_ons_district'})
    .drop_duplicates('local_authority_ons_district')  # added
)
col = col.merge(la_lookup, on='local_authority_ons_district', how='left')

print('Derived columns added and label maps applied.')
print(f'col shape: {col.shape}  cas shape: {cas.shape}  veh shape: {veh.shape}')



Derived columns added and label maps applied.
col shape: (1347870, 55)  cas shape: (1748311, 25)  veh shape: (2469085, 33)


In [11]:

# ── 4. Quick completeness check ───────────────────────────────────────────────
# Sanity check — row counts by year and null rates on key fields before writing output.

print('=== Row counts by year ===')
year_counts = col.groupby(['collision_year', 'provisional']).size().rename('rows').reset_index()
print(year_counts.to_string(index=False))

print('\n=== Null rates — key collision fields ===')
key_fields = ['collision_severity', 'speed_limit', 'road_type', 'urban_or_rural_area',
              'collision_adjusted_severity_serious', 'latitude', 'longitude']
for f in key_fields:
    if f in col.columns:
        pct = col[f].isna().mean() * 100
        print(f'  {f:<42} {pct:.1f}% null')

# Check for duplicate codes in the LA lookup before merging
dupes = la_lookup['local_authority_ons_district'].duplicated().sum()
print(f'Duplicate LA codes in lookup: {dupes}')



=== Row counts by year ===
 collision_year  provisional   rows
           2014        False 146670
           2015        False 140400
           2016        False 136989
           2017        False 130295
           2018        False 122904
           2019        False 117818
           2020        False  91306
           2021        False 101234
           2022        False 106179
           2023        False 104432
           2024        False 101093
           2025         True  48550

=== Null rates — key collision fields ===
  collision_severity                         0.0% null
  speed_limit                                0.0% null
  road_type                                  0.0% null
  urban_or_rural_area                        0.0% null
  collision_adjusted_severity_serious        0.0% null
  latitude                                   0.0% null
  longitude                                  0.0% null
Duplicate LA codes in lookup: 0


In [7]:

# ── 5. Build combined dataset ─────────────────────────────────────────────────
# Casualties joined to vehicles on the composite key collision_index +
# collision_year + vehicle_reference, then collision-level context columns
# brought in from col. One row per casualty, with their associated vehicle
# and collision fields alongside.

# Step 1 — join casualties to vehicles on composite key.
# Left join: every casualty is retained; vehicle fields are NaN where no match
# exists (e.g. pedestrians have no associated vehicle record).
cas_full = cas.merge(
    veh,
    on=['collision_index', 'collision_year', 'vehicle_reference'],
    how='left',
    suffixes=('', '_veh')
)

# Step 2 — bring in collision-level context from col.
# Select only the fields most useful for analysis — avoids a very wide table.
COL_CONTEXT = [
    'collision_index', 'collision_year', 'date', 'month', 'day_of_week', 'day_label',
    'collision_severity', 'ksi', 'fatal',
    'speed_limit', 'road_type', 'road_type_label',
    'junction_detail', 'junction_label',
    'urban_or_rural_area', 'ur_label',
    'light_conditions', 'weather_conditions', 'road_surface_conditions',
    'police_force', 'force_name',
    'local_authority_ons_district', 'la_name',
    'latitude', 'longitude',
    'collision_adjusted_severity_serious', 'collision_injury_based',
    'provisional',
]
# Drop any col_ctx columns already present on cas_full (from the cas or veh load)
# to avoid duplicate _col suffix columns in the output.
cas_full_cols = set(cas_full.columns)
col_ctx_cols  = ['collision_index', 'collision_year'] + [
    c for c in COL_CONTEXT
    if c not in ('collision_index', 'collision_year') and c not in cas_full_cols
]
col_ctx = (col[[c for c in col_ctx_cols if c in col.columns]]
           .drop_duplicates(['collision_index', 'collision_year']))

cas_full = cas_full.merge(
    col_ctx,
    on=['collision_index', 'collision_year'],
    how='left',
    suffixes=('', '_col')
)

print(f'cas_full shape: {cas_full.shape}')
print(f'Rows with no vehicle match (e.g. pedestrians): {cas_full["vehicle_type"].isna().sum():,}')
# Integrity check — every casualty row should link back to a collision.
unmatched = cas_full['collision_severity'].isna().sum()
print(f'Casualties with no collision match (should be 0): {unmatched:,}')



cas_full shape: (1748311, 80)
Rows with no vehicle match (e.g. pedestrians): 7,044
Casualties with no collision match (should be 0): 0


In [ ]:

# ── 6. Write cleaned CSVs ─────────────────────────────────────────────────────
# Four files written to OUTPUT_DIR (configured at the top).
# Existing files are overwritten.

os.makedirs(OUTPUT_DIR, exist_ok=True)

col.to_csv(os.path.join(OUTPUT_DIR, 'collisions_clean.csv'), index=False)
cas.to_csv(os.path.join(OUTPUT_DIR, 'casualties_clean.csv'), index=False)
veh.to_csv(os.path.join(OUTPUT_DIR, 'vehicles_clean.csv'),   index=False)
cas_full.to_csv(os.path.join(OUTPUT_DIR, 'cas_full.csv'),    index=False)

print(f'Written to: {OUTPUT_DIR}')
print(f'  collisions_clean.csv — {len(col):,} rows')
print(f'  casualties_clean.csv — {len(cas):,} rows')
print(f'  vehicles_clean.csv   — {len(veh):,} rows')
print(f'  cas_full.csv         — {len(cas_full):,} rows')
